# Supplier catalogue preflight

I used the WooCommerce sample catalogue and a simulated supplier update to practise checking products and proposed selling prices. AI (OpenAI Codex) helped with the simulated data and parts of the code. This is not paid client work.

## M1 - Inspect the store file

I read the CSV as text so SKUs keep their original characters. An empty CSV field stays an empty string. The first table shows the fields used in this review.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

store = pd.read_csv("data/sample_products.csv", dtype=str, keep_default_na=False)

print("Rows and columns:", store.shape)
store[["ID", "Type", "SKU", "Name", "Parent", "Regular price"]].head()

Rows and columns: (25, 51)


,ID,Type,SKU,Name,Parent,Regular price
0,44,variable,woo-vneck-tee,V-Neck T-Shirt,,
1,45,variable,woo-hoodie,Hoodie,,
2,46,simple,woo-hoodie-with-logo,Hoodie with Logo,,45
3,47,simple,woo-tshirt,T-Shirt,,18
4,48,simple,woo-beanie,Beanie,,20


In [2]:
print(store["Type"].value_counts())

# Show parent products and variations together to understand their relationship.
store.loc[
    store["Type"].isin(["variable", "variation"]),
    ["ID", "Type", "SKU", "Name", "Parent", "Regular price"]
]

Type
simple                           12
variation                         7
variable                          2
simple, downloadable, virtual     2
grouped                           1
external                          1
Name: count, dtype: int64


,ID,Type,SKU,Name,Parent,Regular price
0,44,variable,woo-vneck-tee,V-Neck T-Shirt,,
1,45,variable,woo-hoodie,Hoodie,,
14,76,variation,woo-vneck-tee-red,V-Neck T-Shirt - Red,woo-vneck-tee,20
15,77,variation,woo-vneck-tee-green,V-Neck T-Shirt - Green,woo-vneck-tee,20
16,78,variation,woo-vneck-tee-blue,V-Neck T-Shirt - Blue,woo-vneck-tee,15
17,79,variation,woo-hoodie-red,"Hoodie - Red, No",woo-hoodie,45
18,80,variation,woo-hoodie-green,"Hoodie - Green, No",woo-hoodie,45
19,81,variation,woo-hoodie-blue,"Hoodie - Blue, No",woo-hoodie,45
24,90,variation,woo-hoodie-blue-logo,"Hoodie - Blue, Yes",woo-hoodie,45


In [3]:
included = store["Type"].isin(["simple", "variation"])

store_review = store.loc[included].copy()
excluded_store = store.loc[~included].copy()

print("Total store rows:", len(store))
print("Included rows:", len(store_review))
print("Excluded rows:", len(excluded_store))
print("Counts reconcile:", len(store) == len(store_review) + len(excluded_store))

excluded_store["Type"].value_counts()

Total store rows: 25
Included rows: 19
Excluded rows: 6
Counts reconcile: True


Type
variable                         2
simple, downloadable, virtual    2
grouped                          1
external                         1
Name: count, dtype: int64

I included 12 simple products and 7 variations. I excluded 2 parent products, 2 virtual/downloadable products, 1 grouped product and 1 external product.

The parent rows use `variable`. Their individual versions use `variation`, with their own SKU and price. In this sample, `Parent` contains the parent's SKU.

Column meanings:
- `ID`: the store record identifier.
- `Type`: the kind of product record.
- `SKU`: the product code used for matching.
- `Name`: the product or variation name.
- `Parent`: the parent SKU for a variation in this sample.
- `Regular price`: the current selling price, in sample price units.

The following checks count blank SKUs, repeated nonblank SKUs, outer-space changes and unusable prices. A repeated SKU could make a match ambiguous.


In [4]:
sku_for_check = store_review["SKU"].str.strip()

print("Missing or blank store SKUs:", (sku_for_check.isna() | sku_for_check.eq("")).sum())
print("Duplicate store rows:", (sku_for_check.duplicated(keep=False) & sku_for_check.ne("")).sum())
print("Store SKUs with outer spaces:", store_review["SKU"].ne(sku_for_check).sum())

Missing or blank store SKUs: 0
Duplicate store rows: 0
Store SKUs with outer spaces: 0


In [5]:
prices = pd.to_numeric(store_review["Regular price"], errors="coerce")

print("Missing or non-numeric prices:", prices.isna().sum())
print("Zero or negative prices:", prices.le(0).sum())

Missing or non-numeric prices: 0
Zero or negative prices: 0


## M2 - Make the simulated supplier update

I start with the included store items. `copy()` keeps this work separate from the store table. The renamed columns describe the supplier data, and the index gives each row a text ID starting at 1. `source_store_id` keeps the link to the store while generating the cases.

The next cells use `.loc` to change the four prices, add spaces around the belt SKU and leave the cap price blank. `concat` adds the extra sunglasses entry and the invented shoe brush. The exported supplier table has 21 rows and four columns.


In [6]:
simulation = store_review[["ID", "SKU", "Name", "Regular price"]].copy()

simulation = simulation.rename(columns={
    "ID": "source_store_id",
    "SKU": "supplier_sku",
    "Name": "supplier_name",
    "Regular price": "proposed_price"
}).reset_index(drop=True)

simulation["supplier_row_id"] = (simulation.index + 1).astype(str)

simulation.head()

,source_store_id,supplier_sku,supplier_name,proposed_price,supplier_row_id
0,46,woo-hoodie-with-logo,Hoodie with Logo,45,1
1,47,woo-tshirt,T-Shirt,18,2
2,48,woo-beanie,Beanie,20,3
3,58,woo-belt,Belt,65,4
4,60,woo-cap,Cap,18,5


In [7]:
simulation.loc[simulation["source_store_id"].eq("47"), "proposed_price"] = "20"
simulation.loc[simulation["source_store_id"].eq("48"), "proposed_price"] = "18"
simulation.loc[simulation["source_store_id"].eq("76"), "proposed_price"] = "22"
simulation.loc[simulation["source_store_id"].eq("80"), "proposed_price"] = "42"

simulation.loc[simulation["source_store_id"].eq("58"), "supplier_sku"] = " woo-belt "
simulation.loc[simulation["source_store_id"].eq("60"), "proposed_price"] = ""

In [8]:
extra_sunglasses = simulation.loc[simulation["source_store_id"].eq("62")].copy()
extra_sunglasses["supplier_row_id"] = "20"
extra_sunglasses["proposed_price"] = "95"

new_item = pd.DataFrame([{
    "source_store_id": "",
    "supplier_sku": "sim-new-shoe-brush",
    "supplier_name": "Simulated Shoe Brush",
    "proposed_price": "12",
    "supplier_row_id": "21"
}])

simulation = pd.concat([simulation, extra_sunglasses, new_item], ignore_index=True)

generated_supplier = simulation[[
    "supplier_row_id", "supplier_sku", "supplier_name", "proposed_price"
]].copy()

generated_supplier.head()

,supplier_row_id,supplier_sku,supplier_name,proposed_price
0,1,woo-hoodie-with-logo,Hoodie with Logo,45
1,2,woo-tshirt,T-Shirt,20
2,3,woo-beanie,Beanie,18
3,4,woo-belt,Belt,65
4,5,woo-cap,Cap,


The expected-results table records the intended outcome of each planned case. The dictionary lists the deliberate changes; the loop fills their status, case name and note. These expectations are written before matching and do not come from the later classifier.


In [9]:
expected_generated = pd.DataFrame({
    "supplier_row_id": simulation["supplier_row_id"],
    "expected_status": "exact_match",
    "test_case": "normal_exact_match",
    "expected_note": (
        "Derived from store ID " + simulation["source_store_id"]
        + "; original SKU and selling price unchanged."
    )
})

planned_cases = {
    '2': ('exact_match', 'price_change', 'Derived from store ID 47; original SKU and selling price unchanged. Proposed selling price changed from 18 to 20.'),
    '3': ('exact_match', 'price_change', 'Derived from store ID 48; original SKU and selling price unchanged. Proposed selling price changed from 20 to 18.'),
    '4': ('exact_match', 'whitespace_sku', 'Store ID 58: one leading and trailing space added; price 65 unchanged. Expect a cleaning note.'),
    '5': ('missing_or_invalid_price', 'invalid_price', 'Store ID 60: proposed price deliberately left blank; SKU unchanged.'),
    '6': ('duplicate_supplier_sku', 'duplicate_supplier_sku', 'Store ID 62: original price 90 retained; SKU repeated in supplier row 20 at price 95.'),
    '11': ('exact_match', 'price_change', 'Derived from store ID 76; original SKU and selling price unchanged. Proposed selling price changed from 20 to 22.'),
    '15': ('exact_match', 'price_change', 'Derived from store ID 80; original SKU and selling price unchanged. Proposed selling price changed from 45 to 42.'),
    '20': ('duplicate_supplier_sku', 'duplicate_supplier_sku', 'Store ID 62: extra sunglasses row at 95, conflicting with supplier row 6 at 90.'),
    '21': ('missing_store_sku', 'missing_store_sku', 'Invented item with no source store row; unique absent SKU and valid proposed price 12.'),
}

for row_id, values in planned_cases.items():
    expected_generated.loc[
        expected_generated["supplier_row_id"].eq(row_id),
        ["expected_status", "test_case", "expected_note"]
    ] = list(values)

expected_generated["expected_status"].value_counts()

expected_status
exact_match                 17
duplicate_supplier_sku       2
missing_or_invalid_price     1
missing_store_sku            1
Name: count, dtype: int64

In [10]:
from pathlib import Path

if not Path("data/supplier_update.csv").exists():
    generated_supplier.to_csv("data/supplier_update.csv", index=False)

if not Path("data/expected_results.csv").exists():
    expected_generated.to_csv("data/expected_results.csv", index=False)

print("Generated supplier rows:", len(generated_supplier))
print("Generated expected-result rows:", len(expected_generated))

Generated supplier rows: 21
Generated expected-result rows: 21


## M3 - Prepare the working data

I load the supplier CSV as text. Separate working SKUs trim outer spaces, while the originals stay unchanged. Numeric price columns allow subtraction; `errors="coerce"` turns missing or nonnumeric prices into missing numeric values for checking.

The checks below keep duplicate and invalid rows for review. The cleaning note records the belt SKU change beside its supplier row ID.


In [11]:
supplier = pd.read_csv("data/supplier_update.csv", dtype=str, keep_default_na=False)

store_review["working_sku"] = store_review["SKU"].str.strip()
supplier["working_sku"] = supplier["supplier_sku"].str.strip()

proposed_prices = pd.to_numeric(supplier["proposed_price"], errors="coerce").round(2)
store_review["current_price"] = prices.round(2)
supplier["proposed_price_numeric"] = proposed_prices

print("Supplier rows:", len(supplier))
supplier[["supplier_row_id", "supplier_sku", "proposed_price", "proposed_price_numeric"]].head()

Supplier rows: 21


,supplier_row_id,supplier_sku,proposed_price,proposed_price_numeric
0,1,woo-hoodie-with-logo,45,45.0
1,2,woo-tshirt,20,20.0
2,3,woo-beanie,18,18.0
3,4,woo-belt,65,65.0
4,5,woo-cap,,NaN


In [12]:
blank_sku = supplier["working_sku"].eq("")
duplicate_sku = supplier["working_sku"].duplicated(keep=False) & ~blank_sku

print("Rows with blank SKUs:", blank_sku.sum())
print("Rows with duplicate SKUs:", duplicate_sku.sum())

supplier.loc[
    duplicate_sku,
    ["supplier_row_id", "supplier_sku", "working_sku", "proposed_price"]
]

Rows with blank SKUs: 0
Rows with duplicate SKUs: 2


,supplier_row_id,supplier_sku,working_sku,proposed_price
5,6,woo-sunglasses,woo-sunglasses,90
19,20,woo-sunglasses,woo-sunglasses,95


In [13]:
invalid_price = proposed_prices.isna() | proposed_prices.le(0)
print("Rows with missing or invalid prices:", invalid_price.sum())

supplier.loc[invalid_price, ["supplier_row_id", "supplier_sku", "proposed_price"]]

Rows with missing or invalid prices: 1


,supplier_row_id,supplier_sku,proposed_price
4,5,woo-cap,


In [14]:
supplier["sku_cleaning_note"] = ""

changed_sku = supplier["supplier_sku"].ne(supplier["working_sku"])

supplier.loc[changed_sku, "sku_cleaning_note"] = (
    "Leading/trailing whitespace removed before matching"
)

supplier.loc[
    changed_sku,
    ["supplier_row_id", "supplier_sku", "working_sku", "sku_cleaning_note"]
]

,supplier_row_id,supplier_sku,working_sku,sku_cleaning_note
3,4,woo-belt,woo-belt,Leading/trailing whitespace removed before mat...


In [15]:
print("Store IDs present:", store_review["ID"].ne("").all())
print("Store IDs unique:", store_review["ID"].is_unique)

print("Supplier row IDs present:", supplier["supplier_row_id"].ne("").all())
print("Supplier row IDs unique:", supplier["supplier_row_id"].is_unique)

Store IDs present: True
Store IDs unique: True
Supplier row IDs present: True
Supplier row IDs unique: True


## M4 - Match and classify

These three functions are shared by the main review and the small QA examples. `match_store` keeps supplier rows and avoids joining to ambiguous store SKUs. `classify_rows` applies the brief's status order. `record_other_issues` keeps detectable problems that were not selected as the primary status.

The left join and `many_to_one` check help prevent lost rows or unexpected multiplication. Price differences are proposed price minus current price, calculated only for valid unique exact matches.


In [16]:
def match_store(supplier_data, store_data):
    store_counts = store_data["working_sku"].value_counts()

    unique_store = store_data.loc[
        store_data["working_sku"].ne("")
        & ~store_data["working_sku"].duplicated(keep=False)
    ]

    result = supplier_data.merge(
        unique_store,
        on="working_sku",
        how="left",
        validate="many_to_one",
        indicator=True
    )

    result["duplicate_store_sku"] = (
        result["working_sku"].map(store_counts).gt(1)
    )

    return result

In [17]:
def classify_rows(data, store_data):
    blank_sku = data["working_sku"].eq("")

    duplicate_supplier = (
        data["working_sku"].duplicated(keep=False) & ~blank_sku
    )

    store_counts = store_data["working_sku"].value_counts()
    duplicate_store = data["working_sku"].map(store_counts).gt(1)

    missing_store = data["_merge"].eq("left_only")

    supplier_price_bad = (
        data["proposed_price_numeric"].isna()
        | data["proposed_price_numeric"].le(0)
    )

    store_price_bad = (
        data["_merge"].eq("both")
        & (
            data["current_price"].isna()
            | data["current_price"].le(0)
        )
    )

    return np.select(
        [
            blank_sku,
            duplicate_supplier,
            duplicate_store,
            missing_store,
            supplier_price_bad | store_price_bad
        ],
        [
            "missing_or_invalid_sku",
            "duplicate_supplier_sku",
            "duplicate_store_sku",
            "missing_store_sku",
            "missing_or_invalid_price"
        ],
        default="exact_match"
    )

In [18]:
def record_other_issues(data):
    notes = pd.Series("", index=data.index)

    other_primary_status = data["status"].ne("missing_or_invalid_price")

    supplier_price_bad = (
        data["proposed_price_numeric"].isna()
        | data["proposed_price_numeric"].le(0)
    )

    store_price_bad = (
        data["_merge"].eq("both")
        & (
            data["current_price"].isna()
            | data["current_price"].le(0)
        )
    )

    notes.loc[other_primary_status & supplier_price_bad] = (
        "Proposed price missing or invalid"
    )

    notes.loc[other_primary_status & store_price_bad] += (
        "; Current store price missing or invalid"
    )

    duplicate_supplier = (
        data["working_sku"].duplicated(keep=False)
        & data["working_sku"].ne("")
    )

    duplicate_store = data["duplicate_store_sku"]

    missing_store = (
        data["_merge"].eq("left_only")
        & ~duplicate_store
        & data["working_sku"].ne("")
    )

    checks = [
        (
            duplicate_supplier,
            "duplicate_supplier_sku",
            "Supplier SKU occurs more than once"
        ),
        (
            duplicate_store,
            "duplicate_store_sku",
            "SKU matches multiple store records"
        ),
        (
            missing_store,
            "missing_store_sku",
            "SKU not found in store"
        )
    ]

    for condition, primary_status, explanation in checks:
        affected = condition & data["status"].ne(primary_status)
        notes.loc[affected] += "; " + explanation

    return notes.str.strip("; ")

In [19]:
review = match_store(
    supplier,
    store_review[["ID", "SKU", "Name", "working_sku", "current_price"]]
)

review["status"] = classify_rows(review, store_review)

review["other_issues"] = record_other_issues(review)

review["status"].value_counts()

status
exact_match                 17
duplicate_supplier_sku       2
missing_or_invalid_price     1
missing_store_sku            1
Name: count, dtype: int64

In [20]:
exact_match = review["status"].eq("exact_match")

review["price_difference"] = np.nan

review.loc[exact_match, "price_difference"] = (
    review.loc[exact_match, "proposed_price_numeric"]
    - review.loc[exact_match, "current_price"]
).round(2)

In [21]:
review["price_change_direction"] = ""

review.loc[exact_match, "price_change_direction"] = "unchanged"

review.loc[
    review["price_difference"].gt(0),
    "price_change_direction"
] = "increase"

review.loc[
    review["price_difference"].lt(0),
    "price_change_direction"
] = "decrease"


review[["supplier_row_id", "current_price", "proposed_price_numeric", "price_difference", "price_change_direction", "status"]].head()

,supplier_row_id,current_price,proposed_price_numeric,price_difference,price_change_direction,status
0,1,45.0,45.0,0.0,unchanged,exact_match
1,2,18.0,20.0,2.0,increase,exact_match
2,3,20.0,18.0,-2.0,decrease,exact_match
3,4,65.0,65.0,0.0,unchanged,exact_match
4,5,18.0,NaN,NaN,,missing_or_invalid_price


## M5 - Check the results

I check that each supplier row appears once and compare every primary status with the independently planned expected results. The price checks cover the four planned changes and confirm that exception differences stay blank.

I also manually checked the problem cases, the belt whitespace case and five ordinary matches against the source files. The small examples below test duplicate store SKUs, status priority and secondary issues using the same matching functions.


In [22]:
print("Supplier rows:", len(supplier))
print("Review rows:", len(review))

print("Review row IDs unique:", review["supplier_row_id"].is_unique)

print(
    "Same supplier row IDs:",
    set(supplier["supplier_row_id"]) == set(review["supplier_row_id"])
)

Supplier rows: 21
Review rows: 21
Review row IDs unique: True
Same supplier row IDs: True


In [23]:
expected = pd.read_csv(
    "data/expected_results.csv",
    dtype=str,
    keep_default_na=False
)

qa = expected.merge(
    review[["supplier_row_id", "status"]],
    on="supplier_row_id",
    how="outer",
    validate="one_to_one",
    indicator=True
)

qa["classification_matches"] = (
    qa["_merge"].eq("both")
    & qa["expected_status"].eq(qa["status"])
)

print("Rows checked:", len(qa))
print("Rows passing:", qa["classification_matches"].sum())

qa.loc[
    ~qa["classification_matches"],
    ["supplier_row_id", "expected_status", "status", "_merge"]
]

Rows checked: 21
Rows passing: 21


,supplier_row_id,expected_status,status,_merge


In [24]:
expected_differences = pd.Series({
    "2": 2.0,
    "3": -2.0,
    "11": 2.0,
    "15": -3.0
})

actual_differences = review.set_index("supplier_row_id")["price_difference"]

print(
    "Planned price changes correct:",
    actual_differences.reindex(expected_differences.index)
    .eq(expected_differences)
    .all()
)

print(
    "Exception differences are blank:",
    review.loc[
        review["status"].ne("exact_match"),
        "price_difference"
    ].isna().all()
)

Planned price changes correct: True
Exception differences are blank: True


In [25]:
test_store = pd.DataFrame({
    "working_sku": ["TEST-001", "TEST-001"],
    "current_price": [20, 25]
})

test_supplier = pd.DataFrame({
    "supplier_row_id": ["TEST-ROW-1"],
    "working_sku": ["TEST-001"],
    "proposed_price_numeric": [22]
})

test_review = match_store(test_supplier, test_store)

test_review["status"] = classify_rows(test_review, test_store)

test_review[
    ["supplier_row_id", "duplicate_store_sku", "status"]
]

,supplier_row_id,duplicate_store_sku,status
0,TEST-ROW-1,True,duplicate_store_sku


In [26]:
overlap_store = pd.DataFrame({
    "working_sku": ["TEST-002"],
    "current_price": [20]
})

overlap_supplier = pd.DataFrame({
    "supplier_row_id": ["OVERLAP-1", "OVERLAP-2"],
    "working_sku": ["TEST-002", "TEST-002"],
    "proposed_price_numeric": [np.nan, 22]
})

overlap_review = match_store(overlap_supplier, overlap_store)

overlap_review["status"] = classify_rows(
    overlap_review, overlap_store
)

overlap_review["other_issues"] = record_other_issues(overlap_review)

overlap_review[["supplier_row_id", "status", "other_issues"]]

,supplier_row_id,status,other_issues
0,OVERLAP-1,duplicate_supplier_sku,Proposed price missing or invalid
1,OVERLAP-2,duplicate_supplier_sku,


In [27]:
secondary_supplier = pd.DataFrame({
    "supplier_row_id": ["SECONDARY-1", "SECONDARY-2"],
    "working_sku": ["ABSENT-001", "ABSENT-001"],
    "proposed_price_numeric": [22, 23]
})

secondary_review = match_store(secondary_supplier, overlap_store)

secondary_review["status"] = classify_rows(
    secondary_review, overlap_store
)

secondary_review["other_issues"] = record_other_issues(secondary_review)

secondary_review[["supplier_row_id", "status", "other_issues"]]

,supplier_row_id,status,other_issues
0,SECONDARY-1,duplicate_supplier_sku,SKU not found in store
1,SECONDARY-2,duplicate_supplier_sku,SKU not found in store


## M6 and M7 - Prepare the outputs

I select the columns the operator needs. The dictionaries map status labels to plain reasons and suggested actions. Exact matches with a price increase or decrease still need approval. The renamed headings are for the client-facing files.


In [28]:
review_columns = [
    "supplier_row_id",
    "supplier_sku",
    "supplier_name",
    "SKU",
    "Name",
    "current_price",
    "proposed_price_numeric",
    "price_difference",
    "price_change_direction",
    "status",
    "other_issues",
    "sku_cleaning_note"
]

workbook_review = review[review_columns].copy()

In [29]:
status_reasons = {
    "missing_or_invalid_sku": "SKU is missing or invalid.",
    "duplicate_supplier_sku": "SKU appears more than once in the supplier file.",
    "duplicate_store_sku": "SKU appears more than once in the store review.",
    "missing_store_sku": "SKU was not found in the included store products.",
    "missing_or_invalid_price": "A required price is missing or invalid.",
    "exact_match": "SKU matches one store item and prices are usable."
}

status_actions = {
    "missing_or_invalid_sku": "Confirm the supplier SKU.",
    "duplicate_supplier_sku": "Confirm which supplier entry to use.",
    "duplicate_store_sku": "Confirm which store item is intended.",
    "missing_store_sku": "Check the SKU and whether the item belongs in this review.",
    "missing_or_invalid_price": "Confirm the missing or invalid price.",
    "exact_match": "No price change needed."
}

workbook_review["reason"] = workbook_review["status"].map(status_reasons)

workbook_review["action_needed"] = workbook_review["status"].map(status_actions)

In [30]:
changed_price = (
    workbook_review["status"].eq("exact_match")
    & workbook_review["price_change_direction"].isin(["increase", "decrease"])
)

workbook_review.loc[changed_price, "action_needed"] = (
    "Approve or reject the proposed price change."
)

In [31]:
workbook_review = workbook_review.rename(columns={
    "supplier_row_id": "Supplier row ID",
    "supplier_sku": "Original supplier SKU",
    "supplier_name": "Supplier name",
    "SKU": "Matched store SKU",
    "Name": "Matched store product",
    "current_price": "Current store price",
    "proposed_price_numeric": "Proposed supplier price",
    "price_difference": "Price difference",
    "price_change_direction": "Price change direction",
    "status": "Primary status",
    "other_issues": "Other issues",
    "sku_cleaning_note": "SKU cleaning note",
    "reason": "Reason",
    "action_needed": "Action needed"
})

workbook_review.head()

,Supplier row ID,Original supplier SKU,Supplier name,Matched store SKU,Matched store product,Current store price,Proposed supplier price,Price difference,Price change direction,Primary status,Other issues,SKU cleaning note,Reason,Action needed
0,1,woo-hoodie-with-logo,Hoodie with Logo,woo-hoodie-with-logo,Hoodie with Logo,45.0,45.0,0.0,unchanged,exact_match,,,SKU matches one store item and prices are usable.,No price change needed.
1,2,woo-tshirt,T-Shirt,woo-tshirt,T-Shirt,18.0,20.0,2.0,increase,exact_match,,,SKU matches one store item and prices are usable.,Approve or reject the proposed price change.
2,3,woo-beanie,Beanie,woo-beanie,Beanie,20.0,18.0,-2.0,decrease,exact_match,,,SKU matches one store item and prices are usable.,Approve or reject the proposed price change.
3,4,woo-belt,Belt,woo-belt,Belt,65.0,65.0,0.0,unchanged,exact_match,,Leading/trailing whitespace removed before mat...,SKU matches one store item and prices are usable.,No price change needed.
4,5,woo-cap,Cap,woo-cap,Cap,18.0,NaN,NaN,,missing_or_invalid_price,,,A required price is missing or invalid.,Confirm the missing or invalid price.


In [32]:
summary = pd.DataFrame([
    ["Total supplier rows reviewed", len(review)],
    ["Exact matches", review["status"].eq("exact_match").sum()],
    ["Missing or invalid SKUs", review["status"].eq("missing_or_invalid_sku").sum()],
    ["Missing store SKUs", review["status"].eq("missing_store_sku").sum()],
    ["Duplicate supplier SKUs", review["status"].eq("duplicate_supplier_sku").sum()],
    ["Duplicate store SKUs", review["status"].eq("duplicate_store_sku").sum()],
    ["Missing or invalid prices", review["status"].eq("missing_or_invalid_price").sum()],
    ["Price increases", review["price_change_direction"].eq("increase").sum()],
    ["Price decreases", review["price_change_direction"].eq("decrease").sum()],
    ["Unchanged prices", review["price_change_direction"].eq("unchanged").sum()],
    ["Primary exceptions", review["status"].ne("exact_match").sum()],
    ["Rows with additional issues", review["other_issues"].ne("").sum()]
], columns=["Measure", "Count"])

summary

,Measure,Count
0,Total supplier rows reviewed,21
1,Exact matches,17
2,Missing or invalid SKUs,0
3,Missing store SKUs,1
4,Duplicate supplier SKUs,2
5,Duplicate store SKUs,0
6,Missing or invalid prices,1
7,Price increases,2
8,Price decreases,2
9,Unchanged prices,13


The CSV is exported on each run. Existing simulated input CSVs and the finished Excel workbook are kept. `Path.exists()` is the check that prevents an export from replacing the completed Excel file.

The final workbook was finished and checked in Excel: a Review table with filters, frozen panes at C2, two-decimal prices, wrapped explanations and separate colours for problem rows and price changes. Summary COUNTIF formulas and the written findings were added there. Its calculated counts agreed with this Pandas summary. See the README for rebuilding instructions.


In [33]:
workbook_review.to_csv("outputs/review.csv", index=False)

workbook_path = Path("outputs/supplier_catalogue_review.xlsx")

if not workbook_path.exists():
    with pd.ExcelWriter(workbook_path, engine="openpyxl") as writer:
        summary.to_excel(writer, sheet_name="Summary", index=False)
        workbook_review.to_excel(writer, sheet_name="Review", index=False)
else:
    print("Existing Excel workbook kept. No changes made.")

Existing Excel workbook kept. No changes made.
